# Phase 2 Evaluation — Model H (Hindi) and Model L (Nepali)

Runs every Phase 2 evaluation deliverable in **one GPU session**, for both
models, from the two `checkpoint_best.pt` files already on Drive:

| | Deliverable | Module | Output |
|:--|:--|:--|:--|
| 3 | Perplexity / BPB / bits-per-char on val + test | `src.eval.intrinsic` | `{lang}/eval/intrinsic.json` |
| 4 | Loss and optimisation curves | `src.eval.plot_curves` | `report/figures/*.png` |
| 6 | Samples: greedy + temperature 0.5 / 1.0 / 1.5 | `src.eval.generate` | `{lang}/samples/generations.{json,md}` |
| 5 | BLEU-4, chrF++, ROUGE-L, repetition, Distinct-n | `src.eval.metrics` | `report/figures/generation_metrics.{json,md}` |
| 7 | Attention heatmaps, entropy, mean distance | `src.eval.attention_analysis` | `{lang}/figures/attention_*.png` |

**Runtime: GPU.** Nothing here trains; the whole notebook is ~30-45 minutes,
almost all of it the full-split perplexity pass.

Every artefact is stamped with a provenance block — the command, the git
commit, and a SHA-256 for each input file — so each number in the report can
be traced back to the checkpoint and the split that produced it.


## 1. GPU and environment


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))


## 2. Mount Drive and load the code

Same pattern as the pretraining notebooks: the repository is synced to
`24KaratAu/code/` on Drive and copied into the runtime, so there is no
GitHub token and no `git clone` in a notebook.


In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase2_eval'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}
!ls src/eval/


### Devanagari font

matplotlib's default font has no Devanagari coverage. Without this the
attention heatmaps render every token label as an empty box. The analysis
module detects the missing font and falls back to position indices, but the
figures are far more readable with real tokens on the axes.


In [ ]:
!apt-get install -qq fonts-noto-devanagari > /dev/null 2>&1
!pip install -q pyyaml sentencepiece

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
names = {f.name for f in fm.fontManager.ttflist}
print('Devanagari font available:',
      [n for n in names if 'Devanagari' in n] or 'NONE — labels will be positional')


### Check the inputs are all present

Fail here, loudly, rather than three cells in with a half-written artefact.


In [ ]:
from pathlib import Path

LANGS = ['hindi', 'nepali']
REQUIRED = {}
for lang in LANGS:
    REQUIRED[lang] = {
        'checkpoint': f'{DRIVE_BASE}/{lang}/checkpoints/checkpoint_best.pt',
        'model_config': f'{lang}/configs/model.yaml',
        'tokenizer': f'tokenizers/{lang}/{lang}_bpe_8k.model',
        'test_jsonl': f'{DRIVE_BASE}/{lang}/splits/test.jsonl',
        'val_bin': f'{DRIVE_BASE}/{lang}/tokens/{lang}_val.bin',
        'test_bin': f'{DRIVE_BASE}/{lang}/tokens/{lang}_test.bin',
        'val_sidecar': f'{DRIVE_BASE}/{lang}/tokens/{lang}_val.json',
        'test_sidecar': f'{DRIVE_BASE}/{lang}/tokens/{lang}_test.json',
        'log': f'{lang}/logs/train.jsonl',
    }

missing = []
for lang, paths in REQUIRED.items():
    for name, p in paths.items():
        ok = Path(p).exists()
        size = f'{Path(p).stat().st_size/1e6:>10,.1f} MB' if ok else '   MISSING'
        print(f'{lang:7} {name:13} {size}  {p}')
        if not ok:
            missing.append(f'{lang}/{name}: {p}')
assert not missing, 'missing inputs:\n  ' + '\n  '.join(missing)
print('\nall inputs present')


### Sidecars must post-date the byte-counting fix

An early version of the packer counted the JSON envelope (`doc_id`,
`source`, `text_sha256` — roughly 45% of a record) as text, which understates
bits-per-byte by about 6%. `src.eval.intrinsic` refuses such a sidecar, but
checking here means finding out now rather than after the perplexity pass.


In [ ]:
import json
for lang in LANGS:
    for split in ('val', 'test'):
        s = json.load(open(f'{DRIVE_BASE}/{lang}/tokens/{lang}_{split}.json'))
        fixed = ('n_bytes_jsonl' in s) or ('bytes_repaired_at' in s)
        bpc = s['n_bytes_utf8'] / s['n_chars']
        bpt = s['n_bytes_utf8'] / s['n_tokens_no_eos']
        print(f'{lang:7} {split:5} repaired={fixed}  {bpc:.3f} bytes/char  {bpt:.3f} bytes/token')
        assert fixed, f'{lang}_{split}.json predates the byte-counting fix — re-run pack_tokens --repair-bytes'


## 3. Deliverable 3 — perplexity, bits-per-byte, bits-per-character

Full validation **and** test splits, strided in non-overlapping 512-token
windows so every token is scored exactly once. This is the long cell:
roughly 50M tokens per language.

Perplexity depends on the tokenizer, so it does not compare across two
models with different vocabularies. Bits-per-byte and bits-per-character do,
which is why all three are reported.


### Running the modules


In [ ]:
import shlex, subprocess, sys

def run(module, **kwargs):
    """Run an evaluation module as a subprocess, echoing the exact command first.

    Building the argument list here rather than using a shell magic with
    backslash continuations keeps the calls out of IPython line-continuation
    handling, which is unreliable inside an indented loop. The echoed command is
    the one to paste into the report if a number ever has to be reproduced.

    A list or tuple value expands to several argv entries, which is what the
    nargs="+" options (--logs, --generations) require: passing them as one
    space-joined string would make argparse treat the whole thing as a single
    value and split it at the wrong "=".
    """
    cmd = [sys.executable, "-m", module]
    for k, v in kwargs.items():
        cmd.append(f"--{k.replace('_', '-')}")
        cmd.extend(str(x) for x in v) if isinstance(v, (list, tuple)) else cmd.append(str(v))
    print("\n$ " + " ".join(shlex.quote(c) for c in cmd), flush=True)
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f"{module} failed with exit code {r.returncode}")



In [ ]:
for lang in LANGS:
    run("src.eval.intrinsic",
        checkpoint=f"{DRIVE_BASE}/{lang}/checkpoints/checkpoint_best.pt",
        model_config=f"{lang}/configs/model.yaml",
        tokens_dir=f"{DRIVE_BASE}/{lang}/tokens",
        language=lang,
        splits="val,test",
        out=f"{lang}/eval/intrinsic.json")


## 4. Deliverable 4 — loss and optimisation curves

Drawn from the committed training logs, not from anything recomputed here.


In [ ]:
run("src.eval.plot_curves",
    logs=["hindi=hindi/logs/train.jsonl", "nepali=nepali/logs/train.jsonl"],
    outdir="report/figures")


In [ ]:
from IPython.display import Image, display
for f in ('loss_curves.png', 'val_perplexity.png', 'optimisation_diagnostics.png'):
    display(Image(f'report/figures/{f}'))


## 5. Deliverable 6 — generation

32 prefixes drawn by seeded reservoir sampling from each held-out test split,
32 tokens of prompt, 128 new tokens, under greedy decoding and temperature
0.5 / 1.0 / 1.5. The true continuation is stored next to each generation so
the metrics step needs no second GPU pass.


In [ ]:
for lang in LANGS:
    run("src.eval.generate",
        checkpoint=f"{DRIVE_BASE}/{lang}/checkpoints/checkpoint_best.pt",
        model_config=f"{lang}/configs/model.yaml",
        tokenizer=f"tokenizers/{lang}/{lang}_bpe_8k.model",
        test_jsonl=f"{DRIVE_BASE}/{lang}/splits/test.jsonl",
        language=lang,
        n_prompts=32, prompt_tokens=32, max_new_tokens=128,
        out=f"{lang}/samples/generations.json",
        markdown=f"{lang}/samples/generations.md")


### Eyeball one sample per setting

Automated metrics do not tell you whether the text is coherent. Read it.


In [ ]:
for lang in LANGS:
    g = json.load(open(f'{lang}/samples/generations.json'))
    r = g['records'][0]
    print('=' * 78)
    print(f"{lang.upper()}  prompt: {r['prompt_text']}")
    print(f"  [reference] {r['reference_text'][:220]}")
    for name in g['settings']:
        print(f"  [{name:9}] {r['generations'][name]['text'][:220]}")


## 6. Deliverable 5 — generation metrics

BLEU-4, chrF++, ROUGE-L, repetition rate and Distinct-1/2, each implemented
from its definition. The held-out human continuations are scored as a control
row, so the repetition and diversity figures have a baseline to be read
against instead of hanging in the air.


In [ ]:
run("src.eval.metrics",
    generations=["hindi=hindi/samples/generations.json",
                 "nepali=nepali/samples/generations.json"],
    out="report/figures/generation_metrics.json",
    markdown="report/figures/generation_metrics.md")


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('report/figures/generation_metrics.md').read()))


## 7. Deliverable 7 — attention analysis

Heatmaps from an early, a middle and a late layer over four heads on a real
held-out passage, plus per-head entropy, mean attention distance,
previous-token mass and first-position (sink) mass accumulated over 32
held-out windows — 16k query positions, rather than the single sentence in
the heatmaps.


In [ ]:
for lang in LANGS:
    run("src.eval.attention_analysis",
        checkpoint=f"{DRIVE_BASE}/{lang}/checkpoints/checkpoint_best.pt",
        model_config=f"{lang}/configs/model.yaml",
        tokenizer=f"tokenizers/{lang}/{lang}_bpe_8k.model",
        tokens_dir=f"{DRIVE_BASE}/{lang}/tokens",
        language=lang,
        batch_size=4, n_batches=8,
        display_tokens=40, heads="0,1,2,3",
        figdir=f"{lang}/figures",
        out=f"{lang}/eval/attention_stats.json")


In [ ]:
for lang in LANGS:
    print('=' * 78); print(lang.upper())
    for f in sorted(Path(f'{lang}/figures').glob('attention_*.png')):
        display(Image(str(f)))


## 8. Summary table for the report


In [ ]:
rows = []
for lang in LANGS:
    intr = json.load(open(f'{lang}/eval/intrinsic.json'))
    att  = json.load(open(f'{lang}/eval/attention_stats.json'))
    for split, m in intr['splits'].items():
        rows.append((lang, split, m['cross_entropy_nats'], m['perplexity'],
                     m['bits_per_byte'], m['bits_per_char'],
                     m['chars_per_token'], m['n_tokens_scored']))
    print(f"{lang}: head types " + str(att['classification_counts']))

print()
hdr = f"{'model':8} {'split':6} {'CE':>8} {'PPL':>8} {'BPB':>8} {'bits/char':>10} {'chars/tok':>10} {'tokens':>14}"
print(hdr); print('-' * len(hdr))
for r in rows:
    print(f'{r[0]:8} {r[1]:6} {r[2]:8.4f} {r[3]:8.2f} {r[4]:8.4f} {r[5]:10.4f} {r[6]:10.3f} {r[7]:14,}')


## 9. Push every artefact back to Drive

The runtime is wiped on disconnect. These are copied to Drive and then pulled
into the repository locally and committed — the graders read them from the
branch, not from Drive.


In [ ]:
import shutil, os
DEST = f'{DRIVE_BASE}/eval_outputs'

copied = []
for src in ['report/figures'] + [f'{l}/{d}' for l in LANGS
                                 for d in ('eval', 'samples', 'figures')]:
    if not Path(src).exists():
        continue
    dst = f'{DEST}/{src}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n = len(list(Path(src).rglob('*')))
    copied.append((src, n))
    print(f'{src:24} -> {dst}   ({n} files)')

print(f'\n{sum(n for _, n in copied)} files under {DEST}')
print('Pull them locally with:')
print(f'  rclone copy gdrive:24KaratAu/eval_outputs <repo root> -P')
